In [7]:
# 03_modeling.ipynb
# Notebook para entrenamiento y evaluación de modelos ML y redes neuronales

# -------------------------------
# 1. Importar librerías
# -------------------------------
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from src.data_preprocessing import load_data, clean_data, encode_features, scale_features, split_data
from src.modeling import train_traditional_models, load_traditional_models
from src.nn_modeling import train_neural_network, load_neural_network

# -------------------------------
# 2. Cargar y preparar datos
# -------------------------------
df = load_data("../data/raw/telco_churn.csv")
df = clean_data(df)
df_encoded = encode_features(df)

numeric_cols = df_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols.remove('Churn')

df_scaled = scale_features(df_encoded, numeric_cols)
X_train, X_test, y_train, y_test = split_data(df_scaled)

# -------------------------------
# 3. Entrenar modelos tradicionales
# -------------------------------
print("Entrenando modelos tradicionales...")
traditional_models = train_traditional_models(X_train, y_train)

# -------------------------------
# 4. Entrenar red neuronal
# -------------------------------
print("Entrenando red neuronal...")
nn_model = train_neural_network(X_train, y_train, epochs=30, batch_size=32)

# -------------------------------
# 5. Evaluación de modelos
# -------------------------------
def evaluate_model(model, X_test, y_test, is_neural=False):
    if is_neural:
        y_pred_prob = model.predict(X_test).flatten()
        y_pred = (y_pred_prob > 0.5).astype(int)
    else:
        y_pred = model.predict(X_test)
        y_pred_prob = model.predict_proba(X_test)[:, 1]
    
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_pred_prob)
    }

# Evaluar modelos tradicionales
results = {}
for name, model in traditional_models.items():
    results[name] = evaluate_model(model, X_test, y_test)

# Evaluar red neuronal
results['NeuralNetwork'] = evaluate_model(nn_model, X_test, y_test, is_neural=True)

# Mostrar resultados
results_df = pd.DataFrame(results).T
print("\n===== Resultados de Evaluación =====")
display(results_df)

# -------------------------------
# 6. Guardar resultados (opcional)
# -------------------------------
results_df.to_csv("../outputs/metrics.csv", index=True)


c:\users\danilo\documents\proyectos full\ia\ai-portfolio-danilorivera\proyecto08_telco-churn-prediction\src\data_preprocessing.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
c:\users\danilo\documents\proyectos full\ia\ai-portfolio-danilorivera\proyecto08_telco-churn-prediction\src\data_preprocessing.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. T

Entrenando modelos tradicionales...


c:\Users\Danilo\Documents\Proyectos Full\IA\AI-Portfolio-DaniloRivera\venv\lib\site-packages\xgboost\training.py:183: UserWarning: [15:00:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Entrenando red neuronal...


c:\Users\Danilo\Documents\Proyectos Full\IA\AI-Portfolio-DaniloRivera\venv\lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

===== Resultados de Evaluación =====


,Accuracy,Precision,Recall,F1,ROC_AUC
LogisticRegression,0.796309,0.635514,0.545455,0.587050,0.839453
RandomForest,0.794180,0.645833,0.497326,0.561934,0.827786
XGBoost,0.789212,0.618462,0.537433,0.575107,0.819546
NeuralNetwork,0.781405,0.624060,0.443850,0.518750,0.822230
